# Lesson 02 - मायक्रोसॉफ्ट एजंट फ्रेमवर्क एक्सप्लोर करणे

**मायक्रोसॉफ्ट एजंट फ्रेमवर्क (MAF)** हा एआय एजंट तयार करण्यासाठी एक एकत्रित फ्रेमवर्क आहे. हे चार मुख्य घटकांसह एक स्वच्छ, संयोज्य आर्किटेक्चर प्रदान करते:

- **क्लायंट** – एखाद्या AI मॉडेल एन्डपॉइंटशी कनेक्ट होतो आणि संवाद हाताळतो
- **एजंट** – सूचनांसह आणि टूल परिभाषांसह क्लायंटला वेढतो
- **टूल्स** – एजंटच्या क्षमता वाढवतात अशा कस्टम फंक्शन्स ज्यांना मॉडेल कॉल करू शकते
- **सेशन** – अनेक टर्न संवादासाठी संभाषण इतिहास राखतो

या धड्यात, आपण या संकल्पनांचा उपयोग करून **प्रवास बुकिंग एजंट** तयार करू जो गंतव्य स्थळी उपलब्धता तपासतो.


## सेटअप


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## एजंट फ्रेमवर्क आर्किटेक्चर समजून घेणे

मायक्रोसॉफ्ट एजंट फ्रेमवर्क एक थरबद्ध आर्किटेक्चर अनुसरते:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **क्लायंट** – एक `AzureAIProjectAgentProvider` Azure OpenAI डिप्लॉयमेंटशी जोडतो. हा प्रमाणीकरण, विनंती स्वरूपन, आणि प्रतिसाद पार्सिंग हाताळतो.
2. **एजंट** – क्लायंट कडून `provider.create_agent()` द्वारे तयार केलेला एजंट, जो मॉडेल प्रवेशासह सूचना (सिस्टम प्रॉम्प्ट) आणि उपकरणे एकत्र करतो.
3. **उपकरणे** – `@tool` ने सजवलेल्या पायथन फंक्शन्स जे एजंट क्रिया करण्यासाठी किंवा डेटा मिळवण्यासाठी वापरू शकतो.
4. **सत्र** – एक `AgentSession` ऑब्जेक्ट (जो `agent.create_session()` द्वारे तयार होतो) जे संभाषण इतिहास साठवते, ज्यामुळे एजंट पूर्वीच्या संदर्भाला लक्षात ठेवून बहु-फेर संभाषण करू शकतो.

चला प्रत्येक थर टप्प्याटप्प्याने तयार करूया.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool डेकोरेटरसह साधने जोडणे

साधने एजंटला मजकूर तयार करण्यापलीकडे क्रिया करण्याची परवानगी देतात. `@tool` डेकोरेटर एक सामान्य Python फंक्शन एजंट कॉल करू शकतो अशा प्रकारे रूपांतरित करतो.

महत्वाचे मुद्दे:
- मॉडेल प्रत्येक पॅरामीटर समजून घेण्यासाठी `Annotated[type, "description"]` वापरा.
- डॉकस्ट्रिंग साधनाचे वर्णन बनते जे मॉडेलला दिसते.
- `approval_mode="never_require"` म्हणजे साधन वापरकर्त्याच्या पुष्टीशिवाय ऑटोमॅटिक चालते.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## साधनेंसह एजंट तयार करणे

आता आपण क्लायंट, सूचना आणि साधने एकत्र करून एक एजंट तयार करतो. `सूचना` ही सिस्टम प्रॉम्प्ट म्हणून काम करतात — त्या एजंटची व्यक्तिमत्त्व आणि वर्तन निश्चित करतात.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## सत्रांसह बहु-चरण संभाषणे

एक `AgentSession` (जो `agent.create_session()` द्वारा तयार केला जातो) संभाषणातील सर्व संदेशांचे ट्रॅक ठेवतो. प्रत्येक `agent.run()` कॉलमध्ये एकसारखा सत्र दिल्याने, एजंटला पूर्ण संभाषण इतिहास उपलब्ध होतो आणि तो पूर्वीच्या संदेशांकडे परत पाहू शकतो.

आपण `tools=[check_destination_availability]` देतो जेणेकरून एजंट प्रत्येक टर्नमध्ये आमच्या उपलब्धता तपासणी करणाऱ्याला कॉल करू शकेल.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## सारांश

या धड्यात तुम्ही Microsoft Agent Framework चे चार स्तंभ पाहिले:

| संकल्पना | तुम्ही काय शिकलात |
|---------|------------------|
| **क्लायंट** | `AzureAIProjectAgentProvider` क्रेडेन्शियल-आधारित प्रमाणीकरणासह Azure OpenAI शी कनेक्ट होतो |
| **एजंट** | `provider.create_agent()` मॉडेल कनेक्शनशी सूचना आणि नाव बंडल करतो |
| **टूल्स** | `@tool` डेकोरेटर एजंटसाठी कॉल करण्यासाठी Python फंक्शन्स उघडतो |
| **सेशन** | `agent.create_session()` अनेक फेऱ्यांमध्ये संभाषणाचा इतिहास राखतो |

हे बांधकामाचे घटक एकत्रितपणे एजंट तयार करतात जे नैसर्गिक संभाषण करू शकतात, बाह्य फंक्शन्स कॉल करू शकतात, आणि संदर्भ राखू शकतात — जे पुढील धड्यांतील अधिक प्रगत एजंटिक पद्धतींसाठी पाया आहे.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून भाषांतरित केला आहे. आम्ही अचूकतेसाठी प्रयत्नशील असलो तरी, कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये चुका किंवा अपूर्णता असू शकतात. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाच्या माहितीसाठी व्यावसायिक मानवी भाषांतर करण्याची शिफारस केली जाते. या भाषांतराच्या वापरामुळे निर्माण झालेल्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलागी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
